[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-GITHUB-USERNAME/JAXCode/blob/master/templates/b_27_cross_attention_pure.ipynb)

# 🟡 Medium: Cross-Attention without Flax

*Attention & Transformers*
Problem 23 with an explicit parameter pytree.

### Signature
```python
def init_cross_attention(key, d_model, num_heads):
    ...   # -> {"W_q": {...}, "W_k": {...}, "W_v": {...}, "W_o": {...}}

def apply_cross_attention(params, x_q, x_kv, num_heads):
    ...   # (B, seq_q, d_model), (B, seq_kv, d_model) -> (B, seq_q, d_model)
```

Same pytree as `b_26`: four `(d_model, d_model)` kernels scaled by
`1/sqrt(d_model)`, four zero biases, key split four ways.

### The one line that differs from self-attention
```python
q = W_q(x_q)     # queries from one sequence
k = W_k(x_kv)    # keys and values from the other
v = W_v(x_kv)
```

That is genuinely all of it — which is the point of doing this one right after
`b_26`. If your `apply_mha` was written without naming the batch axis, this is
almost a rename.

### The trap it adds
`seq_q` and `seq_kv` are **different**. The scores are
`(..., H, seq_q, seq_kv)`, the output length comes from `Q`, and softmax runs
over the last axis (the keys). Anything that assumed a square score matrix
breaks here, and a square test case would not notice.

### A property worth checking yourself
Feed the same array as both inputs and you must get exactly self-attention
back. That single assertion catches most wiring mistakes — a swapped `x_q` /
`x_kv`, or `W_k` fed the wrong sequence.

In [ ]:
# Colab setup (no-op when running locally).
# jax-judge is not published on PyPI, so the judge is installed from the
# repo itself. Regenerate with JAXCODE_REPO=you/YourFork to point this at
# your own fork:  JAXCODE_REPO=you/JAXCode make notebooks
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q flax optax')
    get_ipython().run_line_magic(
        'pip', 'install -q git+https://github.com/YOUR-GITHUB-USERNAME/JAXCode.git')
except ImportError:
    pass

In [ ]:
import jax
import jax.numpy as jnp

print("JAX", jax.__version__, "|", jax.devices())

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

import jax
import jax.numpy as jnp


def init_cross_attention(key, d_model, num_heads):
    """Parameter pytree: W_q, W_k, W_v, W_o, each a kernel and a bias."""
    pass  # Replace this


def apply_cross_attention(params, x_q, x_kv, num_heads):
    """Queries from x_q attend over keys/values from x_kv."""
    pass  # Replace this

In [ ]:
# 🔍 Scratch cell — poke at your implementation
import jax
import jax.numpy as jnp

params = init_cross_attention(jax.random.key(0), d_model=8, num_heads=2)

x_q = jax.random.normal(jax.random.key(1), (2, 3, 8))    # 3 queries
x_kv = jax.random.normal(jax.random.key(2), (2, 7, 8))   # 7 keys/values
print("seq_q=3, seq_kv=7 ->", apply_cross_attention(params, x_q, x_kv, 2).shape)

# Same input twice must reduce to self-attention.
x = jax.random.normal(jax.random.key(3), (2, 5, 8))
same = apply_cross_attention(params, x, x, 2)
print("cross(x, x) shape: ", same.shape)
print("that IS self-attention — the best single check of the wiring")

In [ ]:
# ✅ SUBMIT — run this cell to check your solution
from jax_judge import check, hint, solution, status

check("cross_attention_pure")

# hint("cross_attention_pure")      # stuck? nudge without the answer
# solution("cross_attention_pure")  # spoiler: the reference implementation
# status()                          # your dashboard across all problems